# Predição de Preços de Livros

### Web Scraping + Data Science: 

------------------------------------------------------------------------------------------------

# Passo 1 - Importar Bibliotecas

### Bibliotecas importadas

requests: faz requisicoes HTTP para baixar paginas da web

BeautifulSoup: analisa HTML e extrai informacoes de paginas web

pandas: manipula dados em tabelas (DataFrames), essencial para Data Science

numpy: funcoes matematicas e operacoes com arrays (usado pelo pandas/sklearn)

matplotlib: cria graficos e visualizacoes de dados


#### sklearn (scikit-learn): biblioteca de Machine Learning do Python

train_test_split ---> divide dados em treino/teste

RandomForestRegressor ---> algoritmo de ML (floresta de arvores)

LinearRegression ---> algoritmo de ML (regressao linear)

mean_squared_error, r2_score ---> metricas para avaliar modelos



re: expressoes regulares para limpar textos (ex: extrair numeros de precos)
time: funcoes de tempo, usada aqui para pausar entre requisicoes (evita bloqueio)

In [1]:
print("=" * 50)
print("PROJETO: WEB SCRAPING + MACHINE LEARNING")
print("=" * 50)

# 1. IMPORTAR
print("\nImportando bibliotecas...")
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import re
import time
print("✓ Bibliotecas importadas!")

PROJETO: WEB SCRAPING + MACHINE LEARNING

Importando bibliotecas...
✓ Bibliotecas importadas!


# Passo 2 - Web Scraping

### Extrair dados da WEB




In [2]:

"""
    Criar função que acessa o site books.toscrape.com e extrai informacoes dos livros.
    Utilizar parametro max_pages: quantidade de paginas a percorrer (cada pagina tem 20 livros)
    Retornar: DataFrame do pandas com os dados coletados
"""
def raspar_livros(max_paginas=50):

     # URL base do site - {} sera substituido pelo numero da pagina
    url_base = "https://books.toscrape.com/catalogue/page-{}.html"

    # Listas que vão armazenar dados extraidos
    titulos = []    # nomes dos livros
    precos = []    # precos em libras
    avaliacoes = []   # avaliacoes em estrelas (1 a 5)
    
    print(f"\nBuscando {max_paginas} paginas...")
    
    # Loop para percorrer cada pagina do site
    for pagina in range(1, max_paginas + 1):
        try:
            # Monta a URL da pagina atual
            url = url_base.format(pagina)

            # pega o HTML da pagina
            resposta = requests.get(url, timeout=10)

            # BeautifulSoup analisa o HTML baixado
            sopa = BeautifulSoup(resposta.content, 'html.parser')

            # pegar todos os elementos <article> com classe 'product_pod'
            livros = sopa.find_all('article', class_='product_pod')

            
            # Loop para extrair dados de cada livro encontrado na pagina
            for livro in livros:
                
                # EXTRAÇÃO DO TITULO:
                titulos.append(livro.h3.a['title'])

                # EXTRAÇÃO DO PREÇO:
                texto_preco = livro.find('p', class_='price_color').text

                # re.sub remove tudo que nao for digito ou ponto
                precos.append(float(re.sub(r'[^\d.]', '', texto_preco)))

                # EXTRAÇÃO DA AVALIAÇÃO:
                classe_avaliacao = livro.find('p', class_='star-rating')['class'][1]
                mapa_avaliacao = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
                avaliacoes.append(mapa_avaliacao.get(classe_avaliacao, 0))
            
            print(f"✓ Pagina {pagina}: {len(livros)} livros")
            time.sleep(0.3)
            
        except Exception as erro:
            # Se der erro em alguma pagina, mostra e continua para a proxima
            print(f"✗ Erro pagina {pagina}: {erro}")
    

    # Cria DataFrame do pandas: tabela estruturada com os dados
    dados = pd.DataFrame({
        'titulo': titulos,
        'preco': precos,
        'avaliacao': avaliacoes
    })
    
    print(f"\n✓ Total coletado: {len(dados)} livros")
    return dados

# EXECUTA O SCRAPING:
dados = raspar_livros(max_paginas=50)
print(f"\nPreco medio: £{dados['preco'].mean():.2f}")
print(f"Avaliacao media: {dados['avaliacao'].mean():.1f} estrelas")



Buscando 50 paginas...
✓ Pagina 1: 20 livros
✓ Pagina 2: 20 livros
✓ Pagina 3: 20 livros
✓ Pagina 4: 20 livros
✓ Pagina 5: 20 livros
✓ Pagina 6: 20 livros
✓ Pagina 7: 20 livros
✓ Pagina 8: 20 livros
✓ Pagina 9: 20 livros
✓ Pagina 10: 20 livros
✓ Pagina 11: 20 livros
✓ Pagina 12: 20 livros
✓ Pagina 13: 20 livros
✓ Pagina 14: 20 livros
✓ Pagina 15: 20 livros
✓ Pagina 16: 20 livros
✓ Pagina 17: 20 livros
✓ Pagina 18: 20 livros
✓ Pagina 19: 20 livros
✓ Pagina 20: 20 livros
✓ Pagina 21: 20 livros
✓ Pagina 22: 20 livros
✓ Pagina 23: 20 livros
✓ Pagina 24: 20 livros
✓ Pagina 25: 20 livros
✓ Pagina 26: 20 livros
✓ Pagina 27: 20 livros
✓ Pagina 28: 20 livros
✓ Pagina 29: 20 livros
✓ Pagina 30: 20 livros
✓ Pagina 31: 20 livros
✓ Pagina 32: 20 livros
✓ Pagina 33: 20 livros
✓ Pagina 34: 20 livros
✓ Pagina 35: 20 livros
✓ Pagina 36: 20 livros
✓ Pagina 37: 20 livros
✓ Pagina 38: 20 livros
✓ Pagina 39: 20 livros
✓ Pagina 40: 20 livros
✓ Pagina 41: 20 livros
✓ Pagina 42: 20 livros
✓ Pagina 43: 20 liv

# Passo 3 - Machine Learning

### Criar e treinar modelos de previsão




In [3]:

print("\n" + "=" * 50)
print("ETAPA 2: MACHINE LEARNING")
print("=" * 50)

# PREPARACAO DOS DADOS:
# X(maiusculo): o que usamos para prever
# y (minusculo): o que queremos prever

X = dados[['avaliacao']]
y = dados['preco']


# DIVISAO TREINO/TESTE:

# - 80% para treinar o modelo (X_treino, y_treino)
# - 20% para testar/validar (X_teste, y_teste)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTreinando com {len(X_treino)} livros...")
print(f"Testando com {len(X_teste)} livros...")


#   MODELOS DE MACHINE LEARNING:

# Random Forest
floresta = RandomForestRegressor(n_estimators=100, random_state=42)
floresta.fit(X_treino, y_treino)
previsao_floresta = floresta.predict(X_teste)

# Linear Regression
regressao_linear = LinearRegression()
regressao_linear.fit(X_treino, y_treino)
previsao_linear = regressao_linear.predict(X_teste)


ETAPA 2: MACHINE LEARNING

Treinando com 800 livros...
Testando com 200 livros...


# Passo 4 - Resultados de modelos

### Ver qual o resultado mais acertivo






In [4]:

print("\n" + "-" * 30)
print("RESULTADOS:")
print("-" * 30)


# (R2): Mede a eficácia do modelo
# Quanto mais proximo de 1, melhor

r2_floresta = r2_score(y_teste, previsao_floresta) # avalia Random Forest
r2_linear = r2_score(y_teste, previsao_linear) # avalia Regressao Linear

print(f"\nRandom Forest R²: {r2_floresta:.3f}")
print(f"Regressao Linear R²: {r2_linear:.3f}")

#Define o melhor modelo comparando os R2
melhor = "Random Forest" if r2_floresta > r2_linear else "Regressao Linear"
print(f"\n✓ Melhor modelo: {melhor}")


------------------------------
RESULTADOS:
------------------------------

Random Forest R²: -0.013
Regressao Linear R²: -0.007

✓ Melhor modelo: Regressao Linear


# Passo 5 - Previsão com novos dados

### Utilizando novos dados para prever resultados



In [5]:

# Cria um novo dado para testar: livro ficticio com 5 estrelas
novo_livro = pd.DataFrame({'avaliacao': [5]})
# Seleciona o modelo vencedor para fazer a previsao
modelo_final = floresta if r2_floresta > r2_linear else regressao_linear
# .predict() retorna array, pegamos o primeiro (e unico) elemento com [0]
preco_previsto = modelo_final.predict(novo_livro)[0]

print(f"\nLivro com 5 estrelas:")
print(f"Preco previsto: £{preco_previsto:.2f}")




Livro com 5 estrelas:
Preco previsto: £36.03


# Passo 6 - Salvamento dos dados

### Salvar dados em arquivo CSV


In [6]:
import os

# Cria pasta 'dados' se nao existir
pasta_dados = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')
os.makedirs(pasta_dados, exist_ok=True)

# Caminho completo do arquivo
caminho_csv = os.path.join(pasta_dados, 'livros.csv')
dados.to_csv(caminho_csv, index=False)
print(f"\n✓ Dados salvos em: {caminho_csv}")
# =============================================================================
# ENCERRAMENTO
# =============================================================================
print("\n" + "=" * 50)
print("PROJETO CONCLUIDO!")
print("=" * 50)


✓ Dados salvos em: c:\Users\Windows 11\Documents\PROGRAMAÇÃO\Python\Web Scrapping\web-scraping-project\data\livros.csv

PROJETO CONCLUIDO!
